In [2]:
from pathlib import Path

import pandas as pd

# Works whether the notebook server starts in the repository root or notebooks/features.
for project_root in (Path.cwd(), *Path.cwd().parents):
    data_path = project_root / "data/train-2025.csv"
    if data_path.is_file():
        break
else:
    raise FileNotFoundError("Could not find data/raw/train-2025.csv from the current directory.")

data = pd.read_csv(data_path, low_memory=False)
len(data)

1023552

In [3]:
# Flags for economically implausible values; see docs/project-update/features/01-financial-anomalies.md
# Original values are never changed. Run on raw data, before fill_missing (02_missing_values.ipynb), so flags reflect reported values, not imputed ones.
# Non-negative by definition (Riva 10.2, 10.3.2). AR, AP_st, AP_lt are left out: offsetting can make them negative in practice.
NEG_BS_COLS = [
    "asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "liab_lt", "liab_lt_emp",
    "debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "debt_st", "debt_lt",
]


def flag_anomalies(df):
    """Take raw data, return a copy with the 0/1 column neg_bs_flag added. Missing inputs never trigger a flag."""
    x = df[NEG_BS_COLS].apply(pd.to_numeric, errors="coerce")  # blanks, "NA" etc. become NaN; NaN comparisons are False
    df = df.copy()
    df["neg_bs_flag"] = (x < 0).any(axis=1).astype(int)
    return df


In [ ]:
flagged = flag_anomalies(data)

# Check: original columns untouched; flag count matches the doc
pd.testing.assert_frame_equal(flagged[data.columns], data)
assert flagged["neg_bs_flag"].sum() == 5117

# Check: missing-value spellings a holdout may use never trigger a flag
raw = data.head(3).astype(str)
raw.loc[raw.index[0], ["asst_tang_fixed", "debt_st"]] = ["", "NA"]
assert flag_anomalies(raw).iloc[0]["neg_bs_flag"] == 0

flags = ["neg_bs_flag"]
y = flagged["def_date"].notna()
pd.DataFrame({"n": flagged[flags].sum(), "default_rate": [y[flagged[f] == 1].mean().round(4) for f in flags]})

In [ ]:
#which rows are negative when they are not supposed to be
negative_bs_rows = data.loc[(data[NEG_BS_COLS].apply(pd.to_numeric, errors="coerce") < 0).any(axis=1)]
negative_bs_rows["id"].nunique()

n_any_flag = flagged[flags].any(axis=1).sum()
print(f"There are {n_any_flag} of records that has at least 1 wrong column.")